In [1]:
# Load the toolbox and the data
import sys
sys.path.append("../src")

import pandas as pd

from gow_rag.data_loader import load_corpus, load_claims
from gow_rag.preprocessing import make_tokenizer
from gow_rag.sparse_retrieval import SparseRetriever, count_words
from gow_rag.graph_of_words import graph_term_weights
from gow_rag.dense_retrieval import DenseRetriever
from gow_rag.fusion import reciprocal_rank_fusion
from gow_rag.evaluation import evaluate, evaluate_per_query, paired_t_test
from gow_rag.paths import TABLES_DIR, DATA_DIR

tokenizer = make_tokenizer("english", stem=True)
corpus = load_corpus()
claim_sets = {split: load_claims(split) for split in ["validation", "train"]}
print("Papers:", len(corpus), "| Claims:", {s: len(q) for s, (q, _) in claim_sets.items()})

Papers: 5183 | Claims: {'validation': 188, 'train': 505}


In [2]:
# using the word magic engine from notebook 6
doc_ids = corpus["doc_id"].tolist()
doc_tokens = [tokenizer(text) for text in corpus["full_text"]]
doc_lengths = [len(tokens) for tokens in doc_tokens]

bm25 = SparseRetriever(doc_ids, [count_words(t) for t in doc_tokens], doc_lengths, model="bm25")
twidf_log = SparseRetriever(doc_ids, [graph_term_weights(t) for t in doc_tokens], doc_lengths, model="twidf_log")
print("Word-matching engines ready")

Word-matching engines ready


In [3]:
dense = DenseRetriever(
    doc_ids,
    corpus["full_text"].tolist(),
    cache_file=DATA_DIR / "minilm_corpus_vectors.npy",
)
print("Dense engine ready. Vectors:", dense.doc_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Dense engine ready. Vectors: (5183, 384)


In [4]:
# Running everthing and building hybrid
per_query = {}
rows = []
for split, (queries, qrels) in claim_sets.items():
    run_bm25 = bm25.search_all(queries, tokenizer)
    run_tw = twidf_log.search_all(queries, tokenizer)
    run_dense = dense.search_all(queries)

    runs = {
        "BM25": run_bm25,
        "TW-IDF-log": run_tw,
        "Dense (MiniLM)": run_dense,
        "Hybrid: Dense + BM25": reciprocal_rank_fusion([run_dense, run_bm25]),
        "Hybrid: Dense + TW-IDF-log": reciprocal_rank_fusion([run_dense, run_tw]),
    }
    per_query[split] = {name: evaluate_per_query(run, qrels) for name, run in runs.items()}
    for name, run in runs.items():
        rows.append({"Claims": split, "Method": name, **evaluate(run, qrels)})

hybrid_results = pd.DataFrame(rows).set_index(["Claims", "Method"])
hybrid_results

MAP    P@10  Recall@10  MRR@10  \
Claims     Method                                                          
validation BM25                        0.7809  0.0995     0.9235  0.7836   
           TW-IDF-log                  0.7574  0.0973     0.9098  0.7601   
           Dense (MiniLM)              0.7277  0.1000     0.9117  0.7267   
           Hybrid: Dense + BM25        0.7985  0.1032     0.9525  0.8067   
           Hybrid: Dense + TW-IDF-log  0.7996  0.1037     0.9578  0.8107   
train      BM25                        0.8192  0.1024     0.9307  0.8235   
           TW-IDF-log                  0.8099  0.1010     0.9182  0.8133   
           Dense (MiniLM)              0.7804  0.1034     0.9295  0.7848   
           Hybrid: Dense + BM25        0.8294  0.1055     0.9498  0.8332   
           Hybrid: Dense + TW-IDF-log  0.8340  0.1055     0.9498  0.8395   

                                       nDCG@10  
Claims     Method                               
validation BM25                         0.8159  
           TW-IDF-log                   0.7939  
           Dense (MiniLM)               0.7714  
           Hybrid: Dense + BM25         0.8370  
           Hybrid: Dense + TW-IDF-log   0.8401  
train      BM25                         0.8465  
           TW-IDF-log                   0.8358  
           Dense (MiniLM)               0.8181  
           Hybrid: Dense + BM25         0.8595  
           Hybrid: Dense + TW-IDF-log   0.8629

In [5]:
''' we are looking for answers to following question:- 
Does word-matching help the dense model? (Hybrid vs Dense)
Is the dense model better than BM25 alone?
The key one: is the graph a better partner than counting? (the two hybrids against each other)
###'''


comparisons = [
    ("Hybrid: Dense + BM25", "Dense (MiniLM)"),
    ("Hybrid: Dense + TW-IDF-log", "Dense (MiniLM)"),
    ("Dense (MiniLM)", "BM25"),
    ("Hybrid: Dense + TW-IDF-log", "Hybrid: Dense + BM25"),
]
test_rows = []
for split in claim_sets:
    for a, b in comparisons:
        result = paired_t_test(per_query[split][a], per_query[split][b], metric="AP")
        test_rows.append({"Claims": split, "Comparison": f"{a}  vs  {b}", **result,
                          "Significant (p < 0.05)?": "yes" if result["p_value"] < 0.05 else "no"})

hybrid_tests = pd.DataFrame(test_rows)
hybrid_tests["p_value"] = hybrid_tests["p_value"].map(lambda p: f"{p:.2g}")
hybrid_tests

,Claims,Comparison,mean_difference,p_value,Significant (p < 0.05)?
0,validation,Hybrid: Dense + BM25 vs Dense (MiniLM),0.0708,5.5e-05,yes
1,validation,Hybrid: Dense + TW-IDF-log vs Dense (MiniLM),0.0719,0.00042,yes
2,validation,Dense (MiniLM) vs BM25,-0.0532,0.031,yes
3,validation,Hybrid: Dense + TW-IDF-log vs Hybrid: Dense ...,0.0011,0.91,no
4,train,Hybrid: Dense + BM25 vs Dense (MiniLM),0.0490,5.7e-06,yes
5,train,Hybrid: Dense + TW-IDF-log vs Dense (MiniLM),0.0535,2e-06,yes
6,train,Dense (MiniLM) vs BM25,-0.0388,0.012,yes
7,train,Hybrid: Dense + TW-IDF-log vs Hybrid: Dense ...,0.0046,0.44,no


In [6]:
#saving the results
hybrid_results.to_csv(TABLES_DIR / "hybrid_results.csv")
hybrid_tests.to_csv(TABLES_DIR / "hybrid_significance_tests.csv", index=False)
print("Saved 2 tables in:", TABLES_DIR)

Saved 2 tables in: C:\Users\Lenovo\Desktop\Projects for IP Paris Application\Graph of words (Reviewing the paper with dataset)\github_upload\scientific-literature-GOW-RAG\results\tables
